# Laboratorio 8: Misión monedas en R³

**Álgebra Lineal · Universidad Tecnológica de Bolívar**

Este cuaderno organiza las tres etapas del laboratorio por celdas:

1. Análisis matemático en Python.
2. Diseño de nuevas posiciones y videojuego.
3. Nueva base, punto no alcanzable y experimento de control.

Cada actividad incluye una celda de código y una explicación breve.

## Preparación

Ejecute esta celda una sola vez al iniciar el cuaderno. Contiene las
funciones utilizadas en todas las etapas.

In [ ]:
from fractions import Fraction

Vector = tuple[Fraction, ...]


def vector(*components: int) -> Vector:
    """Crea un vector de R3 con componentes exactas."""
    return tuple(Fraction(component) for component in components)


def linear_combination(vectors, coefficients):
    """Calcula a*v1 + b*v2 + c*v3."""
    return tuple(
        coefficients[0] * vectors[0][row]
        + coefficients[1] * vectors[1][row]
        + coefficients[2] * vectors[2][row]
        for row in range(3)
    )


def matrix(vectors):
    """Construye la matriz A = [v1 v2 v3]."""
    return [
        [vectors[column][row] for column in range(3)]
        for row in range(3)
    ]


def determinant3(matrix_a):
    """Calcula el determinante de una matriz 3x3."""
    a, b, c = matrix_a[0]
    d, e, f = matrix_a[1]
    g, h, i = matrix_a[2]
    return (
        a * (e * i - f * h)
        - b * (d * i - f * g)
        + c * (d * h - e * g)
    )


def gauss_jordan(matrix_a, P):
    """Resuelve Ax=P por eliminación de Gauss-Jordan."""
    augmented = [
        row[:] + [P[row_index]]
        for row_index, row in enumerate(matrix_a)
    ]

    pivot_row = 0
    for column in range(3):
        pivot = next(
            (
                row for row in range(pivot_row, 3)
                if augmented[row][column] != 0
            ),
            None,
        )
        if pivot is None:
            continue

        augmented[pivot_row], augmented[pivot] = (
            augmented[pivot],
            augmented[pivot_row],
        )
        pivot_value = augmented[pivot_row][column]
        augmented[pivot_row] = [
            value / pivot_value
            for value in augmented[pivot_row]
        ]

        for row in range(3):
            if row == pivot_row:
                continue
            factor = augmented[row][column]
            augmented[row] = [
                augmented[row][index]
                - factor * augmented[pivot_row][index]
                for index in range(4)
            ]
        pivot_row += 1

    for row in augmented:
        if all(value == 0 for value in row[:3]) and row[3] != 0:
            return augmented, None

    if pivot_row < 3:
        return augmented, None

    return augmented, [augmented[row][3] for row in range(3)]


def integer_nonnegative(solution):
    """Indica si todos los coeficientes son enteros no negativos."""
    return solution is not None and all(
        value.denominator == 1 and value >= 0
        for value in solution
    )


def format_vector(value):
    return "(" + ", ".join(str(component) for component in value) + ")"


def analyze_base(vectors, objectives):
    """Calcula el determinante y clasifica todos los objetivos."""
    matrix_a = matrix(vectors)
    det = determinant3(matrix_a)

    print("\nMatriz A:")
    for row in matrix_a:
        print(format_vector(row))
    print("det(A) =", det)
    print("¿Es una base?", "Sí" if det != 0 else "No")

    for name, position in objectives.items():
        augmented, solution = gauss_jordan(matrix_a, position)
        reachable = integer_nonnegative(solution)
        coefficients = (
            tuple(int(value) for value in solution)
            if reachable
            else solution
        )

        print(f"\n{name}: P = {format_vector(position)}")
        print("Matriz aumentada:")
        for row in augmented:
            print(format_vector(row[:3]) + " | " + str(row[3]))
        print("Solución (a, b, c):", coefficients)
        print(
            "Clasificación:",
            "ALCANZABLE" if reachable else "NO ALCANZABLE",
        )
        if reachable:
            print("Pulsaciones:", coefficients)

# Etapa 1: Análisis matemático y resolución en Python

## Datos iniciales

Los vectores de movimiento son:

- $v_1=(1,1,0)$
- $v_2=(1,0,1)$
- $v_3=(0,1,1)$

Las monedas son:

- $P_1=(3,3,2)$
- $P_2=(3,2,3)$
- $P_3=(1,1,1)$

### Actividades 1 y 2
La siguiente celda construye la matriz, calcula su determinante y resuelve
los sistemas $Ax=P$.

In [ ]:
v1_original = vector(1, 1, 0)
v2_original = vector(1, 0, 1)
v3_original = vector(0, 1, 1)

base_original = (v1_original, v2_original, v3_original)

monedas_originales = {
    "Moneda 1": vector(3, 3, 2),
    "Moneda 2": vector(3, 2, 3),
    "Moneda 3": vector(1, 1, 1),
}

analyze_base(base_original, monedas_originales)

### Actividad 3: Comprobación de soluciones alcanzables

Verifique que cada combinación reconstruya exactamente la posición.

In [ ]:
coeficientes_m1 = (2, 1, 1)
coeficientes_m2 = (1, 2, 1)

print("Moneda 1:", format_vector(
    linear_combination(base_original, coeficientes_m1)
))
print("Moneda 2:", format_vector(
    linear_combination(base_original, coeficientes_m2)
))

### Actividad 4: Ejecución y registro

Capture una imagen de la salida anterior y responda:

1. ¿El conjunto forma una base?
2. ¿Qué monedas son alcanzables?
3. ¿Por qué la Moneda 3 no cumple las restricciones del juego?

# Etapa 2: Nuevas posiciones y exploración del videojuego

## Actividad 5: Comprobación en el notebook

Primero ejecute el bloque de configuración original mostrado debajo y luego
ejecute la celda del motor del juego.

In [ ]:
#@title ⚙️ Configuración del juego (edite esta celda en las Etapas 2 y 3)
# ------------------------------------------------------------------
# VECTORES DE MOVIMIENTO
# Cada vector debe tener tres componentes enteras.
#   v1 se activa con la flecha derecha  (→)
#   v2 se activa con la flecha izquierda (←)
#   v3 se activa con la flecha arriba    (↑)
# ------------------------------------------------------------------
v1 = (1, 1, 0)
v2 = (1, 0, 1)
v3 = (0, 1, 1)

# ------------------------------------------------------------------
# OBJETIVOS (monedas o estrellas)
# Cada objetivo tiene:
#   "nombre": texto que aparece en la escena
#   "pos":    coordenadas enteras (x, y, z)
#   "tipo":   "moneda" (círculo dorado) o "estrella" (rombo azul)
# Puede agregar o quitar objetivos copiando una línea completa.
# ------------------------------------------------------------------
OBJETIVOS = [
    {"nombre": "Moneda 1", "pos": (3, 3, 2), "tipo": "moneda"},
    {"nombre": "Moneda 2", "pos": (3, 2, 3), "tipo": "moneda"},
    {"nombre": "Moneda 3", "pos": (1, 1, 1), "tipo": "moneda"},
]

print("Configuración guardada. Ahora ejecute la celda del juego.")


: 

## Actividad 6: Diseño de nuevas posiciones

Elija al menos tres ternas $(a,b,c)$ enteras no negativas.

Requisitos:

- Una terna debe tener un coeficiente igual a cero.
- Otra terna debe tener una suma mayor o igual a cinco.
- Las ternas deben ser diferentes de las monedas originales.

In [ ]:
nuevas_ternas = (
    (2, 0, 1),
    (1, 2, 2),
    (0, 3, 1),
)

nuevos_objetivos = {}
for i, coeficientes in enumerate(nuevas_ternas, start=1):
    posicion = linear_combination(base_original, coeficientes)
    nuevos_objetivos[f"Objetivo nuevo {i}"] = posicion
    print(coeficientes, "->", format_vector(posicion))

analyze_base(base_original, nuevos_objetivos)

## Actividad 7: Configuración del videojuego para la Etapa 2

Pegue las posiciones obtenidas en la siguiente celda. Puede conservar las
tres monedas originales y agregar nuevas monedas o estrellas.

In [ ]:
# Modifique los objetivos utilizando las posiciones calculadas arriba.
v1 = (1, 1, 0)
v2 = (1, 0, 1)
v3 = (0, 1, 1)

OBJETIVOS = [
    {"nombre": "Moneda 1", "pos": (3, 3, 2), "tipo": "moneda"},
    {"nombre": "Moneda 2", "pos": (3, 2, 3), "tipo": "moneda"},
    {"nombre": "Moneda 3", "pos": (1, 1, 1), "tipo": "moneda"},
    {"nombre": "Objetivo nuevo 1", "pos": (2, 3, 1), "tipo": "estrella"},
    {"nombre": "Objetivo nuevo 2", "pos": (3, 3, 4), "tipo": "moneda"},
    {"nombre": "Objetivo nuevo 3", "pos": (3, 1, 4), "tipo": "moneda"},
]

print("Configuración de la Etapa 2 guardada.")

## Actividad 8: Registro de combinaciones

Ejecute nuevamente la celda del motor del juego y realice lo siguiente:

- Recoja cada nuevo objetivo.
- Alcance una posición de dos formas distintas, si el juego lo permite.
- Capture la escena, el panel de posición y el registro de combinaciones.

# Etapa 3: Nueva base y experimento de control

## Actividades 9 y 10

Proponga una nueva base y calcule al menos tres posiciones alcanzables.
Incluya un punto entero que no pueda alcanzarse.

In [ ]:
nueva_base = (
    vector(1, 0, 1),
    vector(0, 2, 1),
    vector(1, 1, 0),
)

objetivos_nueva_base = {
    "Objetivo A": linear_combination(nueva_base, (2, 1, 0)),
    "Objetivo B": linear_combination(nueva_base, (0, 2, 3)),
    "Objetivo C": linear_combination(nueva_base, (1, 0, 4)),
    "Punto no alcanzable": vector(-1, 1, 1),
}

analyze_base(nueva_base, objetivos_nueva_base)

## Actividad 11: Configuración de la nueva base en el videojuego

In [ ]:
v1 = (1, 0, 1)
v2 = (0, 2, 1)
v3 = (1, 1, 0)

OBJETIVOS = [
    {"nombre": "Objetivo A", "pos": (2, 2, 3), "tipo": "moneda"},
    {"nombre": "Objetivo B", "pos": (3, 7, 2), "tipo": "estrella"},
    {"nombre": "Objetivo C", "pos": (5, 4, 1), "tipo": "moneda"},
    {"nombre": "Punto no alcanzable", "pos": (-1, 1, 1), "tipo": "estrella"},
]

print("Configuración de la Etapa 3 guardada.")

## Experimento de control

Cambia temporalmente $v_3=v_1+v_2$. Los vectores deben volverse linealmente
dependientes y la matriz debe tener determinante cero.

In [ ]:
base_dependiente = (
    nueva_base[0],
    nueva_base[1],
    (
        nueva_base[0][0] + nueva_base[1][0],
        nueva_base[0][1] + nueva_base[1][1],
        nueva_base[0][2] + nueva_base[1][2],
    ),
)

analyze_base(
    base_dependiente,
    {"Punto de prueba": vector(1, 1, 1)},
)

In [ ]:
v1 = (1, 0, 1)
v2 = (0, 2, 1)
v3 = (
    v1[0] + v2[0],
    v1[1] + v2[1],
    v1[2] + v2[2],
)

OBJETIVOS = [
    {"nombre": "Punto de control", "pos": (1, 1, 1), "tipo": "moneda"},
]

print("Experimento de control activado.")

# Motor del videojuego

Ejecute esta celda después de cada configuración:

- Configuración original.
- Configuración de la Etapa 2.
- Configuración de la Etapa 3.
- Configuración del experimento de control.

No modifique el código de esta celda.

In [ ]:
#@title 🎮 Misión Monedas en R³ — ejecute esta celda para jugar { display-mode: "form" }
"""
MISIÓN MONEDAS EN R³ — motor del juego
Videojuego didáctico de Álgebra Lineal para Google Colab.

Lee la configuración (v1, v2, v3 y OBJETIVOS) de la celda anterior.
Si esa celda no se ha ejecutado, usa la configuración original del taller.

Arquitectura
- Toda la lógica (estado, reglas, matemáticas, figura 3D y mensajes)
  está en Python.
- Un fragmento mínimo de JavaScript captura las flechas del teclado y
  los clics, y le pide a Python, a través de google.colab, que ejecute
  la acción. Python responde con la figura y los paneles actualizados.
"""

import json
import math
from fractions import Fraction

import plotly.graph_objects as go
from IPython.display import HTML, JSON, display

try:
    from google.colab import output
    EN_COLAB = True
except ImportError:
    EN_COLAB = False


# ==================================================================
# 1. LECTURA Y VALIDACIÓN DE LA CONFIGURACIÓN
# ==================================================================

_cfg = globals()
_vecs = [tuple(_cfg.get("v1", (1, 1, 0))),
         tuple(_cfg.get("v2", (1, 0, 1))),
         tuple(_cfg.get("v3", (0, 1, 1)))]
_objs = _cfg.get("OBJETIVOS", [
    {"nombre": "Moneda 1", "pos": (3, 3, 2), "tipo": "moneda"},
    {"nombre": "Moneda 2", "pos": (3, 2, 3), "tipo": "moneda"},
    {"nombre": "Moneda 3", "pos": (1, 1, 1), "tipo": "moneda"},
])


def determinante(a, b, c):
    """Determinante de la matriz cuyas COLUMNAS son a, b y c."""
    return (a[0] * (b[1] * c[2] - c[1] * b[2])
            - b[0] * (a[1] * c[2] - c[1] * a[2])
            + c[0] * (a[1] * b[2] - b[1] * a[2]))


def validar_configuracion():
    errores = []
    for i, v in enumerate(_vecs, start=1):
        if len(v) != 3 or not all(isinstance(t, int) for t in v):
            errores.append(f"v{i} = {v} debe tener tres componentes enteras.")
    for o in _objs:
        p = tuple(o.get("pos", ()))
        if len(p) != 3 or not all(isinstance(t, int) for t in p):
            errores.append(f"«{o.get('nombre', '?')}» tiene una posición no válida: {p}.")
        if o.get("tipo", "moneda") not in ("moneda", "estrella"):
            errores.append(f"«{o.get('nombre', '?')}»: el tipo debe ser \"moneda\" o \"estrella\".")
    if not errores and determinante(*_vecs) == 0:
        errores.append(
            f"El determinante de la matriz [v1 v2 v3] es 0: los vectores {_vecs[0]}, "
            f"{_vecs[1]} y {_vecs[2]} son linealmente dependientes y no forman una base "
            "de R³. Proponga otros vectores.")
    return errores


# ==================================================================
# 2. DATOS DEL JUEGO
# ==================================================================

_estilo_teclas = [("→", "flecha derecha", "#E4572E"),
                  ("←", "flecha izquierda", "#2E86AB"),
                  ("↑", "flecha arriba", "#2BA84A")]

VECTORES = {
    f"v{i}": {"vec": _vecs[i - 1], "paso": f"Paso {i}", "simbolo": s,
              "texto_tecla": t, "color": c}
    for i, (s, t, c) in enumerate(_estilo_teclas, start=1)
}

# Objetivos indexados por número. A propósito NO se guarda si son
# alcanzables: el juego lo calcula resolviendo el sistema lineal.
MONEDAS = {i: {"nombre": o["nombre"], "pos": tuple(o["pos"]),
               "tipo": o.get("tipo", "moneda")}
           for i, o in enumerate(_objs, start=1)}

DET = determinante(*_vecs)
COLOR_MONEDA, COLOR_ESTRELLA = "#F5B700", "#3F8EFC"
COLOR_RECOGIDA, COLOR_INALCANZABLE = "#9E9E9E", "#C62828"
VECTORES_NO_NEGATIVOS = all(t >= 0 for v in _vecs for t in v)


# ==================================================================
# 3. MATEMÁTICA
# ==================================================================

def coeficientes(punto):
    """Resuelve exactamente a·v1 + b·v2 + c·v3 = punto (regla de Cramer).

    La solución existe y es única para cualquier punto de R³ porque los
    vectores son linealmente independientes (det ≠ 0).
    """
    a = Fraction(determinante(punto, _vecs[1], _vecs[2]), DET)
    b = Fraction(determinante(_vecs[0], punto, _vecs[2]), DET)
    c = Fraction(determinante(_vecs[0], _vecs[1], punto), DET)
    return a, b, c


def es_alcanzable(punto):
    """Alcanzable en el juego = coeficientes enteros y no negativos."""
    return all(q.denominator == 1 and q >= 0 for q in coeficientes(punto))


def fmt(q):
    return str(q.numerator) if q.denominator == 1 else f"{q.numerator}/{q.denominator}"


def tupla(p):
    return "(" + ", ".join(str(v) for v in p) + ")"


def ecuaciones(punto):
    """Texto del sistema a·v1 + b·v2 + c·v3 = punto, fila por fila."""
    filas = []
    for k, valor in enumerate(punto):
        terminos = []
        for coef, var in zip((v[k] for v in _vecs), "abc"):
            if coef == 0:
                continue
            signo = " − " if coef < 0 else " + "
            mag = "" if abs(coef) == 1 else f"{abs(coef)}"
            terminos.append((signo, f"{mag}{var}"))
        if not terminos:
            texto = "0"
        else:
            texto = ("−" if terminos[0][0] == " − " else "") + terminos[0][1]
            texto += "".join(s + t for s, t in terminos[1:])
        filas.append(f"{texto} = {valor}")
    return ",&nbsp; ".join(filas)


# ==================================================================
# 4. ESTADO DEL JUEGO
# ==================================================================

MENSAJE_INICIAL = ("La exploradora está en el origen (0, 0, 0). Haga clic sobre el "
                   "juego y use las flechas del teclado (o los botones) para moverse.")


def estado_inicial():
    return {
        "pos": (0, 0, 0),
        "cuenta": {"v1": 0, "v2": 0, "v3": 0},  # coeficientes a, b, c del intento
        "camino": [],            # segmentos (inicio, fin, vector usado)
        "mov_intento": 0,
        "mov_total": 0,
        "intento": 1,
        "recogidas": set(),
        "declaradas": set(),
        "registro": [],          # (nombre, posición, a, b, c, intento)
        "mensaje": MENSAJE_INICIAL,
        "tipo": "info",
    }


estado = estado_inicial()


def poner_mensaje(texto, tipo="info"):
    estado["mensaje"] = texto
    estado["tipo"] = tipo


def mision_completa():
    return len(estado["recogidas"]) + len(estado["declaradas"]) == len(MONEDAS)


# ==================================================================
# 5. ACCIONES DEL JUGADOR
# ==================================================================

def mover(clave):
    """Suma a la posición actual el vector asociado a la tecla pulsada."""
    if clave not in VECTORES:
        return respuesta()

    datos = VECTORES[clave]
    inicio = estado["pos"]
    fin = tuple(p + v for p, v in zip(inicio, datos["vec"]))

    estado["pos"] = fin
    estado["camino"].append((inicio, fin, clave))
    estado["cuenta"][clave] += 1
    estado["mov_intento"] += 1
    estado["mov_total"] += 1

    for n, m in MONEDAS.items():
        if fin == m["pos"] and n not in estado["recogidas"]:
            estado["recogidas"].add(n)
            a, b, c = (estado["cuenta"][k] for k in ("v1", "v2", "v3"))
            estado["registro"].append((m["nombre"], fin, a, b, c, estado["intento"]))
            poner_mensaje(
                f"<b>¡{m['nombre']} recogida!</b> Llegó a {tupla(fin)} con la "
                f"combinación lineal {tupla(fin)} = {a}·v1 + {b}·v2 + {c}·v3.",
                "exito")
            return respuesta()

    # Con vectores de componentes ≥ 0 las coordenadas nunca disminuyen:
    # si ya superó todos los objetivos pendientes, se avisa.
    pendientes = [m["pos"] for n, m in MONEDAS.items()
                  if n not in estado["recogidas"] and n not in estado["declaradas"]]
    puede_llegar = any(all(q >= f for q, f in zip(p, fin)) for p in pendientes)

    if VECTORES_NO_NEGATIVOS and pendientes and not puede_llegar:
        poner_mensaje(
            f"Está en {tupla(fin)}. Desde aquí ya no puede llegar a ningún objetivo "
            "pendiente: todos los vectores tienen componentes mayores o iguales que cero, "
            "así que sus coordenadas nunca disminuyen. Pulse «Volver al origen» para "
            "empezar un nuevo intento.", "aviso")
    else:
        poner_mensaje(
            f"{datos['paso']} ({datos['simbolo']}): {tupla(inicio)} + "
            f"{tupla(datos['vec'])} = <b>{tupla(fin)}</b>", "info")
    return respuesta()


def volver_al_origen():
    """Nuevo intento desde el origen. Los objetivos recogidos se conservan."""
    if estado["mov_intento"] > 0:
        estado["intento"] += 1
    estado["pos"] = (0, 0, 0)
    estado["cuenta"] = {"v1": 0, "v2": 0, "v3": 0}
    estado["camino"] = []
    estado["mov_intento"] = 0
    poner_mensaje(f"Intento {estado['intento']}: la exploradora vuelve al origen. "
                  "Los objetivos que ya recogió se conservan.", "info")
    return respuesta()


def declarar_inalcanzable(n):
    """El jugador afirma que un objetivo no se puede alcanzar; el juego lo verifica."""
    n = int(n)
    m = MONEDAS[n]

    if n in estado["recogidas"]:
        poner_mensaje(f"Ya recogió «{m['nombre']}», así que sí es alcanzable.", "aviso")
    elif n in estado["declaradas"]:
        poner_mensaje(explicacion_inalcanzable(n), "descubrimiento")
    elif es_alcanzable(m["pos"]):
        poner_mensaje(
            f"«{m['nombre']}» en {tupla(m['pos'])} <b>sí</b> puede alcanzarse con "
            f"movimientos completos. Pista: plantee a·v1 + b·v2 + c·v3 = "
            f"{tupla(m['pos'])}, escriba el sistema de ecuaciones para a, b y c, "
            "y resuélvalo.", "aviso")
    else:
        estado["declaradas"].add(n)
        poner_mensaje(explicacion_inalcanzable(n), "descubrimiento")
    return respuesta()


def explicacion_inalcanzable(n):
    m = MONEDAS[n]
    p = m["pos"]
    a, b, c = coeficientes(p)

    texto = (
        f"<b>¡Descubrimiento! «{m['nombre']}» en {tupla(p)} no se puede alcanzar con "
        "movimientos completos.</b><br>"
        f"Al plantear a·v1 + b·v2 + c·v3 = {tupla(p)} se obtiene el sistema "
        f"{ecuaciones(p)}, cuya única solución es "
        f"a = {fmt(a)}, b = {fmt(b)}, c = {fmt(c)}.<br>"
        "Esa combinación sí existe en R³, porque {v1, v2, v3} es una base: "
        f"{tupla(p)} = {fmt(a)}·v1 + {fmt(b)}·v2 + {fmt(c)}·v3. "
    )
    if any(q.denominator != 1 for q in (a, b, c)):
        texto += "Pero exige fracciones de paso, y en el juego cada pulsación es un paso completo."
    if any(q < 0 for q in (a, b, c)):
        texto += " Además exige un coeficiente negativo, es decir, retroceder."

    # Argumento de paridad, válido cuando todos los vectores suman lo mismo
    sumas = {sum(v) for v in _vecs}
    if len(sumas) == 1:
        s = sumas.pop()
        if s > 1 and sum(p) % s != 0:
            texto += (
                f"<br>Otra forma de verlo: cada movimiento suma exactamente {s} a "
                f"x + y + z. Desde el origen esa suma siempre es múltiplo de {s}, y en "
                f"{tupla(p)} vale {sum(p)}.")
    return texto


def estado_actual():
    return respuesta()


# ==================================================================
# 6. ESCENA 3D (Plotly)
# ==================================================================

def linea(p, q, color, ancho, nombre=None, leyenda=False, guion="solid"):
    return go.Scatter3d(
        x=[p[0], q[0]], y=[p[1], q[1]], z=[p[2], q[2]], mode="lines",
        line=dict(color=color, width=ancho, dash=guion),
        name=nombre, showlegend=leyenda,
        hoverinfo="name" if nombre else "skip")


def flecha(origen, vec, color, nombre):
    """Vector dibujado como segmento + cono en la punta."""
    punta = tuple(o + v for o, v in zip(origen, vec))
    cono = go.Cone(
        x=[punta[0]], y=[punta[1]], z=[punta[2]],
        u=[vec[0]], v=[vec[1]], w=[vec[2]],
        anchor="tip", sizemode="absolute", sizeref=0.3,
        colorscale=[[0, color], [1, color]], showscale=False, hoverinfo="skip")
    return [linea(origen, punta, color, 8, nombre, leyenda=True), cono]


def exploradora(p, s=1.6):
    """Personaje: una mujer con vestido y cabello largo. Sus pies marcan la posición."""
    x, y, z = p
    piel, vestido, pelo = "#8D5524", "#8E44AD", "#2C1B10"
    trazas = []

    for dx in (-0.06 * s, 0.06 * s):                       # piernas
        trazas.append(linea((x + dx, y, z), (x + dx, y, z + 0.22 * s), piel, 6))

    n, r, z_base, z_cintura = 18, 0.18 * s, z + 0.18 * s, z + 0.5 * s   # falda
    xs = [x + r * math.cos(2 * math.pi * k / n) for k in range(n)] + [x]
    ys = [y + r * math.sin(2 * math.pi * k / n) for k in range(n)] + [y]
    zs = [z_base] * n + [z_cintura]
    trazas.append(go.Mesh3d(
        x=xs, y=ys, z=zs, i=list(range(n)), j=[(k + 1) % n for k in range(n)],
        k=[n] * n, color=vestido, flatshading=True, hoverinfo="skip"))

    trazas.append(linea((x, y, z_cintura), (x, y, z + 0.64 * s), vestido, 12))  # torso
    trazas.append(go.Scatter3d(                                                  # brazos
        x=[x - 0.2 * s, x, x + 0.2 * s], y=[y, y, y],
        z=[z + 0.36 * s, z + 0.6 * s, z + 0.36 * s],
        mode="lines", line=dict(color=piel, width=5), showlegend=False, hoverinfo="skip"))

    for dx in (-0.07 * s, 0.07 * s):                       # cabello largo
        trazas.append(linea((x + dx, y, z + 0.78 * s), (x + 1.4 * dx, y, z + 0.56 * s), pelo, 7))
    trazas.append(go.Scatter3d(                            # cabeza
        x=[x], y=[y], z=[z + 0.74 * s], mode="markers",
        marker=dict(size=15, color=piel), showlegend=False, hoverinfo="skip"))

    trazas.append(go.Scatter3d(                            # punto exacto de la posición
        x=[x], y=[y], z=[z], mode="markers",
        marker=dict(size=5, color="white", line=dict(color="#222", width=2)),
        showlegend=False, hovertemplate=f"Exploradora<br>{tupla(p)}<extra></extra>"))
    trazas.append(go.Scatter3d(
        x=[x], y=[y], z=[z + 1.05 * s], mode="text", text=[f"<b>{tupla(p)}</b>"],
        textfont=dict(size=13, color="#222"), showlegend=False, hoverinfo="skip"))
    return trazas


def construir_figura():
    trazas = []

    for clave, d in VECTORES.items():                      # vectores de referencia
        trazas += flecha((0, 0, 0), d["vec"], d["color"],
                         f"{clave} = {tupla(d['vec'])}  {d['simbolo']}")

    for inicio, fin, clave in estado["camino"]:            # trayectoria del intento
        trazas.append(linea(inicio, fin, VECTORES[clave]["color"], 5))
    if estado["camino"]:
        puntos = [estado["camino"][0][0]] + [f for _, f, _ in estado["camino"]]
        trazas.append(go.Scatter3d(
            x=[q[0] for q in puntos], y=[q[1] for q in puntos], z=[q[2] for q in puntos],
            mode="markers", marker=dict(size=3, color="#555"),
            showlegend=False, hoverinfo="skip"))

    for n, m in MONEDAS.items():                           # monedas y estrellas
        p = m["pos"]
        es_estrella = m["tipo"] == "estrella"
        if n in estado["recogidas"]:
            color, etiqueta = COLOR_RECOGIDA, f"{m['nombre']} ✔"
        elif n in estado["declaradas"]:
            color, etiqueta = COLOR_INALCANZABLE, f"{m['nombre']} ✖"
        else:
            color = COLOR_ESTRELLA if es_estrella else COLOR_MONEDA
            etiqueta = f"{m['nombre']}<br>{tupla(p)}"
        trazas.append(linea(p, (p[0], p[1], 0), "#AAAAAA", 2, guion="dash"))
        trazas.append(go.Scatter3d(
            x=[p[0]], y=[p[1]], z=[p[2]], mode="markers+text",
            marker=dict(size=11, color=color, symbol="diamond" if es_estrella else "circle",
                        line=dict(color="#555", width=2)),
            text=[etiqueta], textposition="top center",
            textfont=dict(size=12, color="#333"), showlegend=False,
            hovertemplate=f"{m['nombre']}<br>{tupla(p)}<extra></extra>"))

    trazas += exploradora(estado["pos"])

    coords = [estado["pos"]] + [m["pos"] for m in MONEDAS.values()] + _vecs
    bajo = min(-0.5, min(min(c) for c in coords) - 1)
    alto = max(5, max(max(c) for c in coords) + 1)
    eje = lambda nombre: dict(
        title=nombre, range=[bajo, alto], dtick=1,
        backgroundcolor="#F7F9FC", gridcolor="#D9DEE7", zerolinecolor="#888")

    fig = go.Figure(data=trazas)
    fig.update_layout(
        scene=dict(xaxis=eje("X"), yaxis=eje("Y"), zaxis=eje("Z"),
                   aspectmode="cube", camera=dict(eye=dict(x=1.6, y=-1.6, z=1.0))),
        height=560, margin=dict(l=0, r=0, t=10, b=0),
        legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)"),
        uirevision="mision", paper_bgcolor="white")
    return fig


# ==================================================================
# 7. PANELES HTML
# ==================================================================

def panel_estado():
    x, y, z = estado["pos"]
    a, b, c = (estado["cuenta"][k] for k in ("v1", "v2", "v3"))

    filas = ""
    for n, m in MONEDAS.items():
        if n in estado["recogidas"]:
            e = "<span style='color:#2E7D32'>recogida ✔</span>"
        elif n in estado["declaradas"]:
            e = "<span style='color:#C62828'>inalcanzable ✖</span>"
        else:
            e = "pendiente"
        filas += f"<tr><td>{m['nombre']}</td><td>{tupla(m['pos'])}</td><td>{e}</td></tr>"

    registro = ""
    if estado["registro"]:
        registro = "<h4 style='margin:10px 0 4px 0'>Registro de combinaciones</h4><table class='tabla'>"
        for nombre, p, ra, rb, rc, it in estado["registro"]:
            registro += (f"<tr><td>{nombre}</td><td>{tupla(p)} = {ra}·v1 + {rb}·v2 + "
                         f"{rc}·v3</td><td>intento {it}</td></tr>")
        registro += "</table>"

    return f"""
    <div class="pos">Posición actual: <b>({x}, {y}, {z})</b></div>
    <div class="comb">= {a}·v1 + {b}·v2 + {c}·v3</div>
    <div class="cont">Movimientos en este intento: <b>{estado['mov_intento']}</b> ·
      Total: <b>{estado['mov_total']}</b> · Intento n.º {estado['intento']}</div>
    <table class="tabla">{filas}</table>{registro}"""


def html_mensaje():
    return f"<div class='msg {estado['tipo']}'>{estado['mensaje']}</div>"


def html_final():
    if not mision_completa():
        return ""
    return f"""
    <div class="final">
      <h3>Misión completada</h3>
      <p>Resolvió todos los objetivos: recogió los alcanzables e identificó los que no lo son.
      Usó {estado['mov_total']} movimientos en {estado['intento']} intento(s).</p>
      <p class="pregunta">Si los tres vectores constituyen una base de R³ y generan todo el
      espacio tridimensional, ¿por qué no podemos alcanzar todas las posiciones utilizando
      únicamente movimientos completos?</p>
      <p>Escriba su respuesta antes de abrir la discusión.</p>
      <details><summary>Abrir discusión</summary>
        <p>«Generar R³» significa que todo punto puede escribirse como a·v1 + b·v2 + c·v3
        con <b>a, b, c números reales</b>. Las reglas del juego restringen esos coeficientes
        a enteros no negativos (0, 1, 2, …). Con esa restricción, las posiciones alcanzables
        son solo algunos puntos aislados del espacio, no todo R³.</p>
        <p>Que {{v1, v2, v3}} sea base garantiza que la combinación existe y es única,
        pero no dice nada sobre si sus coeficientes son enteros o positivos.</p>
        <p>Para seguir pensando:</p>
        <ul>
          <li>Si se permitieran movimientos inversos (coeficientes enteros, positivos o
          negativos), ¿qué puntos con coordenadas enteras se podrían alcanzar?</li>
          <li>El determinante de la matriz [v1 v2 v3] es {DET}. ¿Qué relación ve entre ese
          número y las fracciones que aparecen en los coeficientes?</li>
          <li>¿Con qué tres vectores se podría alcanzar cualquier punto de coordenadas
          enteras no negativas usando solo pasos completos?</li>
        </ul>
      </details>
    </div>"""


def respuesta():
    return JSON({
        "fig": json.loads(construir_figura().to_json()),
        "estado": panel_estado(),
        "mensaje": html_mensaje(),
        "final": html_final(),
    })


# ==================================================================
# 8. INTERFAZ (HTML + puente de teclado)
# ==================================================================

ESTILOS = """
<style>
#mision-r3 { font-family: system-ui, -apple-system, 'Segoe UI', sans-serif; color:#1f2430;
  border:2px solid #d9dee7; border-radius:12px; padding:14px; outline:none; max-width:1150px; }
#mision-r3:focus { border-color:#8E44AD; }
#mision-r3 h2 { margin:0 0 4px 0; font-size:22px; }
#mision-r3 .sub { margin:0 0 10px 0; color:#555; font-size:14px; }
#mision-r3 .foco { font-size:12px; padding:3px 8px; border-radius:10px; background:#eee; display:inline-block; margin-bottom:8px; }
#mision-r3 .grid { display:flex; gap:14px; flex-wrap:wrap; }
#mision-r3 .izq { flex:1 1 560px; min-width:320px; }
#mision-r3 .der { flex:0 1 380px; min-width:300px; font-size:14px; }
#mision-r3 .caja { background:#F7F9FC; border:1px solid #e3e7ee; border-radius:10px; padding:10px; margin-bottom:10px; }
#mision-r3 .caja h4 { margin:0 0 6px 0; font-size:14px; }
#mision-r3 .vec { display:flex; align-items:center; gap:8px; margin:6px 0; }
#mision-r3 .vec button { min-width:46px; font-size:18px; color:white; border:none; border-radius:8px; padding:4px 10px; cursor:pointer; }
#mision-r3 .pos { font-size:20px; }
#mision-r3 .comb { font-family: ui-monospace, monospace; margin:2px 0 6px 0; color:#444; }
#mision-r3 .cont { font-size:13px; color:#444; margin-bottom:6px; }
#mision-r3 .tabla { border-collapse:collapse; width:100%; font-size:13px; }
#mision-r3 .tabla td { border-top:1px solid #e3e7ee; padding:3px 4px; }
#mision-r3 .btn { border:1px solid #c5ccd8; background:white; border-radius:8px; padding:5px 10px; cursor:pointer; margin:2px; font-size:13px; }
#mision-r3 .btn:hover { background:#eef1f6; }
#mision-r3 .msg { border-radius:10px; padding:10px; font-size:14px; line-height:1.45; }
#mision-r3 .msg.info { background:#EAF2F8; }
#mision-r3 .msg.exito { background:#E8F5E9; border-left:5px solid #2E7D32; }
#mision-r3 .msg.aviso { background:#FFF8E1; border-left:5px solid #F9A825; }
#mision-r3 .msg.descubrimiento { background:#F3E5F5; border-left:5px solid #8E44AD; }
#mision-r3 .final { margin-top:12px; background:#1f2430; color:white; border-radius:10px; padding:14px; }
#mision-r3 .final h3 { margin:0 0 6px 0; }
#mision-r3 .final .pregunta { font-size:16px; font-weight:600; background:rgba(255,255,255,0.08); padding:10px; border-radius:8px; }
#mision-r3 .final summary { cursor:pointer; font-weight:600; }
.mr3-error { font-family: system-ui, sans-serif; background:#FDECEA; border-left:5px solid #C62828; padding:12px; border-radius:8px; }
</style>
"""

PLANTILLA = """
<div id="mision-r3" tabindex="0">
  <h2>Misión Monedas en R³</h2>
  <p class="sub">Recorra R³ y recoja los objetivos. Cada pulsación suma un vector
  completo a su posición: <b>posición = a·v1 + b·v2 + c·v3</b>, con a, b, c enteros ≥ 0.
  No hay movimientos inversos ni fraccionarios.</p>
  <span class="foco" id="mr3-foco">Haga clic aquí para activar el teclado</span>
  <div class="grid">
    <div class="izq">
      <div id="mr3-grafico" style="height:560px;">Cargando escena 3D…</div>
    </div>
    <div class="der">
      <div class="caja"><h4>Movimientos disponibles</h4>__VECTORES__</div>
      <div class="caja" id="mr3-estado"></div>
      <div id="mr3-mensaje"></div>
      <div class="caja" style="margin-top:10px;">
        <h4>Herramientas</h4>
        <button class="btn" id="mr3-origen">⟲ Volver al origen (nuevo intento)</button>
        <p style="margin:8px 0 4px 0;">Si concluye que un objetivo no se puede alcanzar con
        estas reglas, declárelo. El juego verificará su afirmación.</p>
        __DECLARAR__
      </div>
    </div>
  </div>
  <div id="mr3-final"></div>
</div>
<script>
(function () {
  const raiz = document.getElementById('mision-r3');
  const foco = document.getElementById('mr3-foco');
  let ocupado = false, primera = true;

  async function llamar(nombre, args) {
    if (ocupado) return;
    ocupado = true;
    try {
      const r = await google.colab.kernel.invokeFunction(nombre, args, {});
      pintar(r.data['application/json']);
    } catch (err) {
      document.getElementById('mr3-mensaje').innerHTML =
        "<div class='msg aviso'>No hay conexión con Python. Vuelva a ejecutar la celda.</div>";
    } finally {
      ocupado = false;
    }
  }

  function pintar(d) {
    if (primera) { document.getElementById('mr3-grafico').innerHTML = ''; primera = false; }
    Plotly.react('mr3-grafico', d.fig.data, d.fig.layout, {displaylogo: false, responsive: true});
    document.getElementById('mr3-estado').innerHTML = d.estado;
    document.getElementById('mr3-mensaje').innerHTML = d.mensaje;
    document.getElementById('mr3-final').innerHTML = d.final;
  }

  // Cada pulsación es un solo movimiento (se ignora la tecla mantenida).
  const teclas = {ArrowRight: 'v1', ArrowLeft: 'v2', ArrowUp: 'v3'};
  document.addEventListener('keydown', function (e) {
    if (teclas[e.key]) {
      e.preventDefault();
      if (!e.repeat) llamar('mision.mover', [teclas[e.key]]);
    }
  });

  raiz.querySelectorAll('[data-mover]').forEach(b =>
    b.addEventListener('click', () => { llamar('mision.mover', [b.dataset.mover]); raiz.focus(); }));
  raiz.querySelectorAll('[data-declarar]').forEach(b =>
    b.addEventListener('click', () => { llamar('mision.declarar', [b.dataset.declarar]); raiz.focus(); }));
  document.getElementById('mr3-origen').addEventListener('click',
    () => { llamar('mision.origen', []); raiz.focus(); });

  raiz.addEventListener('focusin', () => { foco.textContent = 'Teclado activo: use → ← ↑'; foco.style.background = '#E8F5E9'; });
  raiz.addEventListener('focusout', () => { foco.textContent = 'Haga clic aquí para activar el teclado'; foco.style.background = '#eee'; });

  const s = document.createElement('script');
  s.src = 'https://cdn.plot.ly/plotly-2.35.2.min.js';
  s.onload = () => { llamar('mision.estado', []); raiz.focus(); };
  document.head.appendChild(s);
})();
</script>
"""


def panel_vectores():
    html = ""
    for clave, d in VECTORES.items():
        html += (
            f"<div class='vec'><button data-mover='{clave}' style='background:{d['color']}'>"
            f"{d['simbolo']}</button><div><b style='color:{d['color']}'>{d['paso']}:</b> "
            f"{clave} = {tupla(d['vec'])}<br><small>Tecla: {d['texto_tecla']} "
            f"({d['simbolo']})</small></div></div>")
    return html


def botones_declarar():
    return "".join(f"<button class='btn' data-declarar='{n}'>{m['nombre']} es inalcanzable</button>"
                   for n, m in MONEDAS.items())


# ==================================================================
# 9. ARRANQUE
# ==================================================================

_errores = validar_configuracion()
if _errores:
    display(HTML(ESTILOS + "<div class='mr3-error'><b>Revise la configuración:</b><ul>"
                 + "".join(f"<li>{e}</li>" for e in _errores) + "</ul></div>"))
elif EN_COLAB:
    output.register_callback("mision.mover", mover)
    output.register_callback("mision.origen", volver_al_origen)
    output.register_callback("mision.declarar", declarar_inalcanzable)
    output.register_callback("mision.estado", estado_actual)
    display(HTML(ESTILOS + PLANTILLA.replace("__VECTORES__", panel_vectores())
                 .replace("__DECLARAR__", botones_declarar())))
else:
    print("Este juego está diseñado para Google Colab: allí se conectan el teclado y la "
          "escena 3D. Abra el cuaderno en https://colab.research.google.com y ejecute las celdas.")


: 

# Recopilación de resultados

Complete en el informe:

| Dato | Resultado |
|---|---|
| $\det(A)$ original | |
| Base original | Sí / No |
| Moneda 1 $(a,b,c)$ | |
| Moneda 2 $(a,b,c)$ | |
| Moneda 3 $(a,b,c)$ | |
| Nuevas posiciones Etapa 2 | |
| Nueva matriz $B$ | |
| $\det(B)$ | |
| Punto no alcanzable | |
| Determinante en el control | |

## Preguntas finales

1. ¿Por qué la Moneda 3 no puede recogerse aunque los vectores formen una base?
2. ¿Por qué no todas las posiciones pueden alcanzarse con coeficientes enteros no negativos?
3. ¿Qué garantiza la independencia lineal sobre los coeficientes de una posición?
4. ¿Cambia la posición final al modificar el orden de las pulsaciones?
5. ¿Qué conjunto genera $v_1$, $v_2$ y $v_3$ cuando son linealmente dependientes?
6. Redacte al menos dos conclusiones sobre combinaciones lineales, independencia lineal y bases.